# 06 · Model comparison and error analysis
Owners: **Member 3** (comparison) and **Member 4** (error analysis). Everyone reviews.

Run this **after** notebooks 02–05 have written their outputs to `results/<model>/`. It needs no
GPU: it only reads predictions. Everything produced here goes into the *Results and Discussion*
and *Error Analysis* sections of the report.

## 0 · Setup
Run this cell first. On **Google Colab** it clones the repo, installs requirements and copies the
data from your Google Drive to the (much faster) local Colab disk. Locally it does nothing special.

**Before the first run:** put `Train.csv`, `Test.csv`, `SampleSubmission.csv` and `Audio.zip`
(downloaded from Zindi) in a Google Drive folder called `swahili_audio`, and set `REPO_URL`.

In [ ]:
import os, sys, shutil, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_URL = "https://github.com/YOUR-GROUP/swahili-audio-sequential.git"   # <-- edit once
    REPO_DIR = Path("/content/swahili-audio-sequential")
    DRIVE_DATA = Path("/content/drive/MyDrive/swahili_audio")
    LOCAL_DATA = Path("/content/data")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if not (LOCAL_DATA / ".ready").exists():
        LOCAL_DATA.mkdir(parents=True, exist_ok=True)
        for f in DRIVE_DATA.glob("*.csv"):
            shutil.copy(f, LOCAL_DATA)
        subprocess.run(["unzip", "-q", "-o", str(DRIVE_DATA / "Audio.zip"), "-d", str(LOCAL_DATA)], check=True)
        (LOCAL_DATA / ".ready").touch()
    os.environ["DATA_DIR"] = str(LOCAL_DATA)
    os.environ["CACHE_DIR"] = str(DRIVE_DATA / "cache")          # features persist between sessions
    os.environ["OUTPUT_BACKUP_DIR"] = str(DRIVE_DATA / "outputs")  # results backed up to Drive
else:
    if Path.cwd().name == "notebooks":
        os.chdir(Path.cwd().parent)
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

In [ ]:
import json
import itertools
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa.display
from IPython.display import Audio, display
from sklearn.metrics import confusion_matrix

from src import config as C
from src.data import get_splits, label_encoder
from src.evaluate import mcnemar, show_log, expected_calibration_error
from src.features import load_audio, logmel

sns.set_theme(style="whitegrid")
OUT = C.RESULTS_DIR / "comparison"
OUT.mkdir(exist_ok=True)
ORDER = ["baseline_svm", "baseline_dtw", "cnn_logmel", "birnn_attention", "wav2vec2_transformer"]
KIND = {"baseline_svm": "Baseline (order-free)", "baseline_dtw": "Baseline (alignment)",
        "cnn_logmel": "Neural: local patterns", "birnn_attention": "Neural: recurrent + attention",
        "wav2vec2_transformer": "Neural: pretrained self-attention"}
keys = [k for k in ORDER if (C.RESULTS_DIR / k / "metrics.json").exists()]
print("Models found:", keys)
metrics = {k: json.loads((C.RESULTS_DIR / k / "metrics.json").read_text()) for k in keys}
preds = {k: pd.read_csv(C.RESULTS_DIR / k / "predictions.csv") for k in keys}
names = {k: metrics[k]["model"] for k in keys}
df = get_splits()
classes, to_id = label_encoder(df.label)

## 1 · Main comparison table (held-out test split)

In [ ]:
cols = ["log_loss", "accuracy", "macro_f1", "top2_accuracy", "roc_auc_ovr_macro", "ece", "n_trainable_params"]
table = pd.DataFrame(metrics).T.reindex(keys)
table.insert(0, "approach", [KIND[k] for k in keys])
table = table[["model", "approach"] + [c for c in cols if c in table.columns]]
table.to_csv(OUT / "comparison_table.csv")
(OUT / "comparison_table.md").write_text(table.round(4).to_markdown())
table.round(4)

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
for a, m, better in zip(ax, ["log_loss", "accuracy", "macro_f1"], ["lower", "higher", "higher"]):
    a.barh([names[k] for k in keys], table[m].astype(float), color=[f"C{i}" for i in range(len(keys))])
    a.set_title(f"{m} ({better} is better)")
    a.invert_yaxis()
fig.tight_layout(); fig.savefig(OUT / "comparison_bars.png", dpi=150); plt.show()

## 2 · Experiment progression (all logged experiments, validation split)

In [ ]:
log = show_log()
log.to_csv(OUT / "all_experiments.csv", index=False)
fig, ax = plt.subplots(figsize=(12, 4))
for i, (m, g) in enumerate(log.groupby("model", sort=False)):
    ax.plot(g.exp_id, g.val_log_loss, "o-", label=m, color=f"C{i}")
ax.set_ylabel("validation log loss"); ax.set_title("Experiment progression per model (lower is better)")
ax.tick_params(axis="x", rotation=60); ax.legend(fontsize=8); fig.tight_layout()
fig.savefig(OUT / "experiment_progression.png", dpi=150); plt.show()
log[["exp_id", "member", "model", "description", "val_log_loss", "val_accuracy", "val_macro_f1",
     "train_seconds", "observation"]]

## 3 · Per-class F1 across models: which words are hard for whom?

In [ ]:
f1 = pd.DataFrame({names[k]: pd.read_csv(C.RESULTS_DIR / k / "per_class.csv", index_col=0)["f1-score"]
                   for k in keys})
f1.to_csv(OUT / "per_class_f1.csv")
fig, ax = plt.subplots(figsize=(1.6 * len(keys) + 3, 6))
sns.heatmap(f1, annot=True, fmt=".2f", cmap="RdYlGn", vmin=max(0, f1.values.min() - .05), vmax=1, ax=ax)
ax.set_title("Per-class F1 on the test split"); fig.tight_layout()
fig.savefig(OUT / "per_class_f1.png", dpi=150); plt.show()

## 4 · Confusion matrices side by side and the most confused word pairs

In [ ]:
fig, axes = plt.subplots(1, len(keys), figsize=(5 * len(keys), 4.6))
axes = np.atleast_1d(axes)
confused = []
for ax, k in zip(axes, keys):
    p = preds[k]
    cm = confusion_matrix(p.true, p.pred, labels=classes)
    sns.heatmap(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues", vmin=0, vmax=1, cbar=False, ax=ax,
                xticklabels=classes, yticklabels=classes)
    ax.set_title(names[k], fontsize=10); ax.tick_params(labelsize=6)
    cm_off = cm.copy()
    np.fill_diagonal(cm_off, 0)
    off = pd.DataFrame(cm_off, index=classes, columns=classes)
    for (t, pr), n in off.stack().sort_values(ascending=False).head(5).items():
        if n:
            confused.append({"model": names[k], "true": t, "predicted": pr, "count": int(n)})
fig.tight_layout(); fig.savefig(OUT / "confusions_all_models.png", dpi=150); plt.show()
confused = pd.DataFrame(confused)
confused.to_csv(OUT / "most_confused_pairs.csv", index=False)
confused

> **Discussion prompt:** Are the most confused pairs the same pairs flagged as acoustically similar in
> EDA section 8 (`results/eda/eda_summary.json → most_similar_pairs`)? Do the order-free SVM's
> confusions disappear in the sequential models? That is direct evidence that temporal order matters.

## 5 · Are the differences statistically significant? (exact McNemar test)

In [ ]:
y_true = preds[keys[0]].true.values
mc = pd.DataFrame(index=[names[k] for k in keys], columns=[names[k] for k in keys], dtype=float)
for a, b in itertools.combinations(keys, 2):
    assert (preds[a].file.values == preds[b].file.values).all()
    r = mcnemar(y_true, preds[a].pred.values, preds[b].pred.values)
    mc.loc[names[a], names[b]] = mc.loc[names[b], names[a]] = r["p_value"]
mc.to_csv(OUT / "mcnemar_pvalues.csv")
mc.round(4)

## 6 · Calibration: why log loss and accuracy can disagree

In [ ]:
fig, ax = plt.subplots(figsize=(5.5, 5))
ax.plot([0, 1], [0, 1], "k--", lw=.8)
for i, k in enumerate(keys):
    p = preds[k]
    P = p[[f"p_{c}" for c in classes]].values
    y = np.array([to_id[t] for t in p.true])
    conf, corr = P.max(1), P.argmax(1) == y
    bins = np.linspace(0, 1, 11)
    xs, ys = [], []
    for lo, hi in zip(bins[:-1], bins[1:]):
        m = (conf > lo) & (conf <= hi)
        if m.sum() >= 5:
            xs.append(conf[m].mean()); ys.append(corr[m].mean())
    ax.plot(xs, ys, "o-", ms=3, label=f"{names[k]} (ECE {expected_calibration_error(y, P):.3f})")
ax.set_xlabel("confidence"); ax.set_ylabel("accuracy"); ax.legend(fontsize=7); ax.set_title("Reliability")
fig.tight_layout(); fig.savefig(OUT / "reliability_all.png", dpi=150); plt.show()

## 7 · Error analysis: do errors concentrate in particular recordings?
Join every test prediction with the per-file audio statistics from the EDA (duration, silence, SNR
proxy, loudness, clipping).

In [ ]:
stats = pd.read_csv(C.RESULTS_DIR / "eda" / "audio_stats.csv")
stats["file"] = stats.path.map(lambda s: s.split("/")[-1])
base = preds[keys[0]][["file", "true"]].copy()
base["file_key"] = base.file.map(lambda s: s.split("/")[-1])
for k in keys:
    base[f"correct_{k}"] = (preds[k].pred.values == preds[k].true.values)
base = base.merge(stats.drop(columns=["path"]), left_on="file_key", right_on="file", how="left",
                  suffixes=("", "_s"))
base["n_models_wrong"] = (~base[[f"correct_{k}" for k in keys]]).sum(1)

fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
for i, k in enumerate(keys):
    for a, col, bins in zip(ax, ["snr_est_db", "trimmed_duration_s"], [5, 5]):
        q = pd.qcut(base[col], bins, duplicates="drop")
        acc = base.groupby(q, observed=True)[f"correct_{k}"].mean()
        a.plot(range(len(acc)), acc.values, "o-", label=names[k], color=f"C{i}")
        a.set_xticks(range(len(acc))); a.set_xticklabels([f"{iv.left:.2f}–{iv.right:.2f}" for iv in acc.index],
                                                         rotation=30, fontsize=7)
ax[0].set_title("Accuracy by SNR-proxy quintile (dB)"); ax[1].set_title("Accuracy by spoken-duration quintile (s)")
ax[0].set_ylabel("accuracy"); ax[1].legend(fontsize=7)
fig.tight_layout(); fig.savefig(OUT / "accuracy_by_snr_duration.png", dpi=150); plt.show()

In [ ]:
cmp = base.groupby(base.n_models_wrong >= max(1, len(keys) - 1))[
    ["snr_est_db", "trimmed_duration_s", "silence_ratio", "rms_db", "clip_frac"]].median()
cmp.index = ["mostly correct", "wrong for (almost) all models"]
cmp.to_csv(OUT / "hard_vs_easy_audio_stats.csv")
print("Median audio properties of universally hard clips vs the rest:")
cmp.round(3)

## 8 · Listen to the hardest clips
Clips that (almost) every model gets wrong. Listen and categorise each one: *noisy*, *truncated/
near-silent*, *mispronounced or different word*, *likely mislabelled*, or *genuinely ambiguous*. Record
the categories in the table below (column `category`): this manual analysis is the core of the
error-analysis section.

In [ ]:
hard = base.sort_values("n_models_wrong", ascending=False).head(12)
paths = df.set_index(df.file.map(lambda s: s.split("/")[-1])).path
rows = []
for _, r in hard.head(6).iterrows():
    guesses = {names[k]: preds[k].loc[preds[k].file == r.file, "pred"].iloc[0] for k in keys}
    print(f"\n{r.file_key} | true = '{r.true}' | wrong for {r.n_models_wrong}/{len(keys)} models | "
          f"SNR~{r.snr_est_db:.1f} dB | speech {r.trimmed_duration_s:.2f}s\n   predictions: {guesses}")
    p = paths.get(r.file_key)
    if p is not None:
        y = load_audio(p)
        fig, ax = plt.subplots(figsize=(6, 1.8))
        librosa.display.specshow(logmel(y), sr=C.SR, hop_length=C.HOP_LENGTH, x_axis="time", ax=ax, cmap="magma")
        ax.set_title(f"true '{r.true}'", fontsize=9); ax.set_yticks([]); plt.show()
        display(Audio(y, rate=C.SR))
    rows.append({"file": r.file_key, "true": r.true, "n_models_wrong": r.n_models_wrong, **guesses,
                 "snr_est_db": r.snr_est_db, "speech_s": r.trimmed_duration_s, "category": ""})
pd.DataFrame(rows).to_csv(OUT / "hardest_clips_to_categorise.csv", index=False)

## 9 · Confidently wrong predictions of the best model (label-noise candidates)
Log loss punishes these the most. Cross-check with `results/eda/flagged_recordings.csv`.

In [ ]:
best_key = min(keys, key=lambda k: metrics[k]["log_loss"])
p = preds[best_key]
cw = p[p.true != p.pred].sort_values("confidence", ascending=False).head(10)
flagged = pd.read_csv(C.RESULTS_DIR / "eda" / "flagged_recordings.csv")
cw["flagged_in_eda"] = cw.file.map(lambda s: s.split("/")[-1]).isin(flagged.file.map(lambda s: s.split("/")[-1]))
cw.to_csv(OUT / "confidently_wrong_best_model.csv", index=False)
print(f"Best model by test log loss: {names[best_key]}")
cw[["file", "true", "pred", "confidence", "p_true", "flagged_in_eda"]]

## 10 · Cost vs performance

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
for i, k in enumerate(keys):
    n = metrics[k].get("n_trainable_params")
    if n:
        ax.scatter(n, metrics[k]["log_loss"], s=60, color=f"C{i}", label=names[k])
ax.set_xscale("log"); ax.set_xlabel("trainable parameters (log scale)"); ax.set_ylabel("test log loss")
ax.legend(fontsize=7); ax.set_title("Model size vs performance"); fig.tight_layout()
fig.savefig(OUT / "params_vs_logloss.png", dpi=150); plt.show()

## 11 · (Exploratory) Do the models make complementary errors?
Averaging the probabilities of the three neural models. This is **not** used to pick a model; it
only suggests future work.

In [ ]:
neural = [k for k in ["cnn_logmel", "birnn_attention", "wav2vec2_transformer"] if k in keys]
if len(neural) >= 2:
    from src.evaluate import compute_metrics
    P = np.mean([preds[k][[f"p_{c}" for c in classes]].values for k in neural], axis=0)
    y = np.array([to_id[t] for t in preds[neural[0]].true])
    ens = compute_metrics(y, P, len(classes))
    (OUT / "ensemble_exploratory.json").write_text(json.dumps(ens, indent=2))
    print("Mean-probability ensemble of", neural, "->", {k: round(v, 4) for k, v in ens.items()})

In [ ]:
from src.utils import persist_outputs
persist_outputs()